# [기초-실습] 통계 101×데이터 분석: (7장) 싱관과 회귀

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## ⚙️ 환경 준비

### 1단계 · 한글 폰트 설치

- 그래프에 한글이 깨지지 않도록 나눔 폰트를 설치합니다.

- 실행 후 **[런타임] - [세션 다시 시작]**을 한 번 눌러야 폰트가 적용됩니다.

In [ ]:
# 구글 코랩 환경에서 한글 폰트 설치 및 설정하기
# 필요시 아래 코드 실행 후, [런타임] - [세션 다시 시작] 후 셀을 다시 실행하세요.
!pip install statsmodels
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

### 2단계 · 라이브러리 불러오기

- 이번 실습에서 쓰는 도구입니다. **한 번만 실행**해 두면 끝까지 사용합니다.

- `smf`는 회귀식을 `'y ~ x'` 형태로 쓰는 모듈, `het_breuschpagan`은 등분산 검정 함수입니다.

In [ ]:
# 파이썬 라이브러리 및 모듈 가져오기
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'NanumGothic'  # 기본 폰트 설정
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지

## 문제 1 · 펭귄의 부리는 길수록 두꺼울까?

`난이도 하` · `예상 15분`

**📖 상황**

- 남극 팔머 기지에서 측정한 펭귄 344마리의 신체 데이터(`penguins`)를 사용합니다.

- 부리 길이(`bill_length_mm`)와 부리 두께(`bill_depth_mm`)가 함께 어떻게 움직이는지 봅니다.

- 그런데 이 데이터에는 **세 종(Adelie·Chinstrap·Gentoo)이 섞여 있습니다.**
  전체를 뭉쳐서 볼 때와 집단을 나눠서 볼 때 **결론이 정반대로 뒤집히는 일**이 실제로 일어납니다.

**🎯 이 문제로 배우는 것**

- 상관계수를 계산하기 **전에** 산점도를 먼저 그려야 하는 이유를, 숫자로 직접 확인합니다.

In [ ]:
# 문제 1 · 데이터 준비 — 실행만 하세요
penguins = sns.load_dataset('penguins')

print("데이터 크기:", penguins.shape)
print("\n종별 개체 수:")
print(penguins['species'].value_counts())
print("\n결측치 개수:")
print(penguins.isna().sum())

penguins.head()

### Q1 · 산점도로 두 변수의 관계를 그려 봅시다

- 가로축 `bill_length_mm`, 세로축 `bill_depth_mm`로 산점도를 그리세요.

- 그래프 제목과 축 라벨을 한글로 알아보기 쉽게 설정하세요.

- `💡 힌트` `sns.scatterplot(data=..., x=..., y=...)` / `plt.title()`, `plt.xlabel()`, `plt.ylabel()`

In [ ]:
# 문제 1 · Q1
# 여기에 코드를 작성해주세요.

plt.figure(figsize=(8, 6))
sns.scatterplot(data=penguins, x='bill_length_mm', y='bill_depth_mm', hue='species')

plt.title('펭귄 부리 길이와 부리 깊이의 관계')
plt.xlabel('부리 길이 (mm)')
plt.ylabel('부리 깊이 (mm)')
plt.tight_layout()

### Q2 · 종별로 색을 나눠 같은 그림을 다시 그려 봅시다

- Q1과 똑같은 산점도에 펭귄의 종(`species`)별로 점의 색을 다르게 칠하세요.

- Q1의 그림과 나란히 놓고 무엇이 달라 보이는지 관찰하세요.

- `💡 힌트` `sns.scatterplot()`에 `hue` 인수를 추가해 보세요.

In [ ]:
# 문제 1 · Q2
# 여기에 코드를 작성해주세요.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Q1: 종 구분 없는 산점도
sns.scatterplot(data=penguins, x='bill_length_mm', y='bill_depth_mm', ax=axes[0])
axes[0].set_title('Q1. 종 구분 없음')
axes[0].set_xlabel('부리 길이 (mm)')
axes[0].set_ylabel('부리 깊이 (mm)')

# Q2: 종별 색 구분
sns.scatterplot(data=penguins, x='bill_length_mm', y='bill_depth_mm', hue='species', ax=axes[1])
axes[1].set_title('Q2. 종(species)별 색 구분')
axes[1].set_xlabel('부리 길이 (mm)')
axes[1].set_ylabel('부리 깊이 (mm)')

plt.tight_layout()

### Q3 · 상관계수를 '전체'와 '종별'로 각각 구해 비교해 봅시다

- 두 변수의 상관계수를 **전체 데이터**에 대해 계산하세요.

- 같은 상관계수를 **종별로 나누어** 계산하세요.

- 두 결과의 **부호**를 비교하세요.

- `💡 힌트` 전체는 `df[['A','B']].dropna().corr()`,
  종별은 `df.groupby('species')[['A','B']].corr()`를 활용해 보세요.

In [ ]:
# 문제 1 · Q3
# 여기에 코드를 작성해주세요.
# 전체 데이터에 대한 상관계수
corr_all = penguins[['bill_length_mm', 'bill_depth_mm']].dropna().corr()
print("전체 상관계수:")
print(corr_all)

# 종별 상관계수
corr_by_species = penguins.groupby('species')[['bill_length_mm', 'bill_depth_mm']].corr()
print("\n종별 상관계수:")
print(corr_by_species)

### 💬 정리 · 결과를 말로 설명해 보기

- Q1의 산점도에서 두 변수는 어떤 방향의 관계로 보였나요? Q3의 전체 상관계수와 부호가 일치하나요?

- Q3의 종별 상관계수는 전체와 부호가 같나요, 다른가요?

- **같은 데이터인데 전체와 집단별 결론이 정반대로 나온 이유**를 설명해 보세요. 무엇이 이 착시를 만들고 있나요?

- 만약 이 데이터에 '종' 정보가 없었다면 우리는 어떤 잘못된 결론을 내렸을까요?

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- 부호가 뒤집히는 열쇠는 **종마다 부리의 '기본 크기'가 다르다**는 데 있습니다. 종별 평균이 어디에 놓여 있는지를 Q2의 그림에서 확인해 보세요.

- "전체를 보면 음, 나누면 양"은 두 변수 사이에 **숨은 집단 변수**가 끼어 있을 때 나타납니다. 이 현상에는 이름이 붙어 있고, 다음 시간 인과추론에서 다시 만납니다.

- 결론의 형태를 이렇게 나눠 써 보면 정확해집니다 — "종을 무시하면 ○○, 종을 고려하면 ○○".

</details>

In [ ]:
# 문제 1 · 정리
# 여기에 의견을 작성해주세요.

어떤 방향인지 애매할 정도로 흩어져있었음 전체와 부호는 일치한거 같음

종별 상관계수는 양수, 전체는 음수로 다름

핵심은 종마다 부리의 '기본 크기'(평균 위치)가 다르다는 겁니다.

종 내부에서는: 부리가 길수록 깊이도 깊어지는 자연스러운 생물학적 관계(양의 상관)가 존재합니다.
종 간에는: Gentoo는 부리 길이는 길지만 깊이는 얕은 쪽에 위치하고, Adelie는 반대로 부리는 짧지만 깊이는 깊은 쪽에 위치합니다. 즉 세 종이 산점도 상에서 서로 다른 대각선 위치에 군집해 있는 겁니다.
이 종 간 위치 차이(군집 간 배치)가 종 내부의 진짜 관계보다 시각적·수치적으로 더 크게 작용해서, 전체를 하나로 합치면 마치 음의 관계처럼 보이는 착시를 만듭니다.

즉, '종'이라는 숨은 변수(교란변수)가 두 변수 사이에 끼어들어 있어서, 이를 무시하고 전체를 뭉쳐서 보면 실제와 반대의 결론에 도달하게 되는 겁니다.

"펭귄의 부리가 길수록 부리 깊이는 얕아진다"는 잘못된 결론을 내렸을 겁니다. 실제로는 정반대로, 같은 종 안에서는 부리가 길수록 깊이도 깊어지는 게 진짜 관계입니다. '종'이라는 집단 변수를 놓치면 개별 집단 내의 참된 관계를 완전히 반대로 해석하게 되는 셈이죠. 이게 바로 심슨의 역설이 실무에서 위험한 이유입니다 — 숨은 집단 변수가 있는 줄 모르고 전체 데이터만 보면, 방향까지 틀린 결론을 내릴 수 있습니다.

"종을 무시하면 부리 길이와 깊이는 약한 음의 관계(r = -0.235)로 보이지만, 종을 고려하면 각 종 내부에서는 뚜렷한 양의 관계(r = 0.39~0.65)로 나타난다. 이는 '종'이라는 숨은 집단 변수가 두 변수 사이의 진짜 관계를 가리고 있었기 때문이며, 이 현상을 심슨의 역설이라 부른다."

## 문제 2 · 날개가 길면 몸무게도 무거울까? — 관계를 숫자로

`난이도 하` · `예상 20분`

**📖 상황**

- 문제 1에서 "숫자 전에 그림"을 배웠습니다. 이제 관계를 **숫자 하나**로 요약해 봅니다.

- 이번에는 **날개 길이(`flipper_length_mm`)와 몸무게(`body_mass_g`)**를 봅니다.
  이 두 변수는 **문제 5까지 계속 사용**하며, 상관 → 회귀 → 검정 → 진단으로 이어집니다.

- **피어슨 상관계수(r)**는 관계의 방향(부호)과 강도(절대값)를 −1~+1로 나타냅니다.
  다만 피어슨은 **두 변수가 정규분포라는 전제**에 기대는 모수적 방법입니다.

**🎯 이 문제로 배우는 것**

- 정규성 점검 → 피어슨(r과 p-값) → 스피어만 비교의 순서로, **강의에서 배운 절차 그대로** 상관을 구합니다.

In [ ]:
# 문제 2 · 데이터 준비 — 실행만 하세요
df_corr = penguins[['flipper_length_mm', 'body_mass_g']].dropna()

print("결측치 제거 전:", len(penguins), "행")
print("결측치 제거 후:", len(df_corr), "행")

# 두 변수의 관계를 그림으로 먼저 확인 (문제 1의 교훈!)
sns.scatterplot(data=df_corr, x='flipper_length_mm', y='body_mass_g', alpha=0.6)
plt.title("날개 길이와 몸무게의 관계")
plt.xlabel("날개 길이 (mm)")
plt.ylabel("몸무게 (g)")
plt.show()

### Q1 · 피어슨의 전제인 정규성을 점검해 봅시다

- `flipper_length_mm`과 `body_mass_g` **각각**에 대해 정규성 검정(Shapiro-Wilk)을 수행하세요.

- 유의수준 0.05를 기준으로 '정규성을 기각하는지'까지 함께 출력하세요.

- `💡 힌트` `stats.shapiro(데이터)`는 통계량과 p-값을 함께 돌려줍니다.

In [ ]:
# 문제 2 · Q1
# 여기에 코드를 작성해주세요.
alpha = 0.05

# flipper_length_mm 정규성 검정
stat_flipper, p_flipper = stats.shapiro(df_corr['flipper_length_mm'])
print("flipper_length_mm")
print(f"  통계량: {stat_flipper:.4f}, p-값: {p_flipper:.4f}")
if p_flipper < alpha:
    print(f"  p-값이 {alpha}보다 작으므로 정규성을 기각한다 (정규분포를 따르지 않는다)")
else:
    print(f"  p-값이 {alpha}보다 크므로 정규성을 기각하지 못한다 (정규분포를 따른다고 볼 수 있다)")

print()

# body_mass_g 정규성 검정
stat_mass, p_mass = stats.shapiro(df_corr['body_mass_g'])
print("body_mass_g")
print(f"  통계량: {stat_mass:.4f}, p-값: {p_mass:.4f}")
if p_mass < alpha:
    print(f"  p-값이 {alpha}보다 작으므로 정규성을 기각한다 (정규분포를 따르지 않는다)")
else:
    print(f"  p-값이 {alpha}보다 크므로 정규성을 기각하지 못한다 (정규분포를 따른다고 볼 수 있다)")

### Q2 · 피어슨 상관계수와 p-값을 함께 구해 봅시다

- 상관계수만이 아니라 **p-값도 함께** 구하세요. 상관계수도 가설검정의 대상입니다.

- 결과는 반드시 `r`, `p_value`라는 이름으로 저장하세요. **문제 3·4에서 이어서 사용합니다.**

- `💡 힌트` `r, p_value = stats.pearsonr(x, y)` — 귀무가설은 "모집단의 상관계수가 0이다"입니다.

In [ ]:
# 문제 2 · Q2
# 여기에 코드를 작성해주세요.
r, p_value = stats.pearsonr(df_corr['flipper_length_mm'], df_corr['body_mass_g'])

print(f"피어슨 상관계수 r = {r:.4f}")
print(f"p-값 = {p_value:.10f}")

alpha = 0.05
if p_value < alpha:
    print(f"\np-값이 {alpha}보다 작으므로 귀무가설을 기각한다")
    print("→ 모집단에서 두 변수의 상관계수는 0이 아니다 (유의한 상관관계가 있다)")
else:
    print(f"\np-값이 {alpha}보다 크므로 귀무가설을 기각하지 못한다")
    print("→ 두 변수 사이에 유의한 상관관계가 있다고 보기 어렵다")

### Q3 · 스피어만 상관계수를 구해 피어슨과 비교해 봅시다

- 스피어만 상관계수(ρ)를 구하세요.

- Q2의 피어슨 값과 **나란히 출력**해 두 값의 차이를 눈으로 확인하세요.

- `💡 힌트` `stats.spearmanr(x, y)`

In [ ]:
# 문제 2 · Q3
# 여기에 코드를 작성해주세요.
rho, p_value_spearman = stats.spearmanr(df_corr['flipper_length_mm'], df_corr['body_mass_g'])

print(f"피어슨 상관계수  r   = {r:.4f}")
print(f"스피어만 상관계수 rho = {rho:.4f}")
print(f"두 값의 차이       = {abs(r - rho):.4f}")


### Q4 · 세 결과를 종합해 관계를 문장으로 정리해 봅시다

- Q1~Q3의 결과를 근거로, 오른쪽 빈칸을 채워 관계를 정리하세요.

- 숫자만 적지 말고 **그 숫자가 뜻하는 바**를 함께 쓰세요.

In [ ]:
# 문제 2 · Q4
# 정규성 검정 결과 (피어슨의 전제가 지켜졌나요?): 아니요 둘 다 정규성 만족하지 않음
# 피어슨 r 값과 p-값: r = 0.8712, p-값 ≈ 0 (< 0.05) 
    # 표본에서 관측된 강한 양의 상관은 우연이 아니며, 모집단에서도
#   두 변수 사이에 상관관계가 존재한다고 볼 수 있다.
# 스피어만 ρ 값: ρ = 0.8400
#   피어슨(0.8712)과의 차이가 0.03에 불과해, 정규성이 깨졌음에도
#   피어슨 결과를 신뢰할 만하다는 것을 뒷받침한다.
# 부호(+/-)가 뜻하는 관계의 방향:  둘 다 양수(+). 날개 길이가 길어질수록 몸무게도 함께 무거워지는
#   양의 방향의 관계이다.
# 절대값의 크기가 뜻하는 관계의 강도: 0.84~0.87은 일반적인 해석 기준에서 '매우 강한' 상관에 해당한다.


### 💬 정리 · 결과를 말로 설명해 보기

- Q1에서 정규성이 기각되었습니다. 그렇다면 피어슨 상관계수를 쓰면 안 되는 걸까요? Q2와 Q3의 값을 비교해서 판단해 보세요.

- 피어슨과 스피어만이 비슷하게 나왔다는 것은 무엇을 뜻할까요? 만약 두 값이 크게 달랐다면 어떤 상황을 의심해야 할까요?

- Q2의 p-값은 무엇에 대한 검정인가요? "상관계수가 크다"와 "상관계수가 유의하다"는 같은 말인가요?

- '날개가 길어지는 것이 몸무게를 무겁게 만드는 **원인**이다'라고 결론 내릴 수 있을까요? **교란변수 / 역인과 / 우연** 세 가지로 나누어 설명해 보세요.

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- 정규성이 깨졌을 때 확인할 것은 "쓰면 안 되나"가 아니라 **"결론이 바뀌는가"**입니다. 두 계수가 거의 같다면 전제 위배의 영향이 작다는 뜻이고, 크게 다르면 이상치나 비선형을 의심합니다.

- p-값은 관계의 **크기**가 아니라 **존재**에 대한 답입니다. 표본이 커지면 아주 작은 상관도 유의해집니다 — 두 질문을 분리해서 답해 보세요.

- 인과를 부정하는 세 갈래를 이 데이터에 대입해 보세요. 교란변수 후보는 문제 1에서 이미 만났습니다.

</details>

In [ ]:
# 문제 2 · 정리
# 여기에 의견을 작성해주세요.

안 되는 건 아닙니다. 이론적으로는 피어슨이 정규성을 전제하지만, 표본 크기가 충분히 크면(여기선 n=342) 어느 정도 정규성이 깨져도 결과가 크게 왜곡되지 않는 것으로 알려져 있습니다. 실제로 Q2(피어슨 r=0.8712)와 Q3(스피어만 ρ=0.8400)의 값을 비교해보면 차이가 0.03에 불과합니다. 정규성 가정이 필요 없는 스피어만이 피어슨과 거의 같은 결론을 내렸다는 건, 정규성 위반이 이 경우엔 결과를 실질적으로 왜곡시키지 않았다는 걸 실증적으로 보여줍니다. 그래서 "정규성 기각 → 피어슨 자동 폐기"가 아니라, "정규성 기각 → 스피어만과 비교 → 일치하면 피어슨 결과를 신뢰"라는 절차가 맞습니다.

비슷하다는 건 두 변수 사이의 관계가 선형적이고, 이상치의 영향을 크게 받지 않는 안정적인 관계라는 뜻입니다. 피어슨은 선형 관계와 정규성에 민감하고 이상치에 취약한 반면, 스피어만은 순위만 사용하므로 단조 관계(꼭 직선이 아니어도 됨)를 포착하고 이상치에 강건합니다. 두 값이 비슷하다는 건 이런 차이가 문제되지 않을 만큼 데이터가 "깔끔하다"는 신호입니다.
만약 두 값이 크게 달랐다면 다음을 의심해야 합니다:

이상치의 존재: 피어슨이 소수의 극단값에 과도하게 끌려갔을 가능성
비선형(하지만 단조적인) 관계: 예를 들어 로그형, 지수형 관계라면 스피어만이 더 잘 잡아내고 피어슨은 과소평가
이런 경우 피어슨 값 하나만 보고 관계의 강도를 판단하면 오해할 수 있습니다

Q2의 p-값은 "모집단에서 두 변수의 상관계수가 0이다"라는 귀무가설에 대한 검정입니다. 즉 "우리가 관측한 정도의 상관관계가, 실제로는 아무 관계가 없는 모집단에서 우연히 나올 확률이 얼마나 되는가"를 나타냅니다.

"크다"와 "유의하다"는 다른 말입니다.

유의하다: 그 상관관계가 0이 아니라고 확신할 수 있다는 뜻 (우연이 아니라는 것)
크다(강하다): 관계의 실질적인 강도, 즉 얼마나 밀접한 관계인지

p-값은 표본 크기에 크게 좌우됩니다. 표본이 아주 크면 r=0.05처럼 아주 약한 상관도 유의하게(p<0.05) 나올 수 있고, 반대로 표본이 작으면 r=0.5처럼 꽤 강한 상관도 유의하지 않게 나올 수 있습니다. 지금 경우는 r=0.87로 강도 자체도 크고, p-값도 유의해서 둘 다 만족하는 사례입니다.

    할 수 없습니다. 상관관계는 인과관계를 증명하지 않습니다. 세 가지 대안을 짚어봐야 합니다.

교란변수(confounding): 예를 들어 '펭귄의 종'이나 '나이/성숙도'가 날개 길이와 몸무게 둘 다에 동시에 영향을 줄 수 있습니다. 종이 클수록(예: Gentoo) 날개도 길고 몸무게도 무거운 것이지, 날개가 몸무게를 직접 늘린 게 아닐 수 있습니다. (문제 1에서 본 '종'이라는 숨은 변수가 여기서도 등장할 가능성이 있습니다.)
역인과(reverse causation): 생물학적으로는 말이 안 되지만, 논리적으로는 "몸무게가 많이 나가는 개체가 날개도 더 크게 자란다"는 반대 방향의 인과 가능성도 이론적으로 배제할 수 없습니다. (물론 이 경우엔 생물학적 발달 과정상 둘 다 성장의 결과일 가능성이 더 큽니다.)
우연: p-값이 극히 작고 r=0.87로 매우 강하기 때문에, 이 정도로 강한 관계가 순전히 우연히 나타났을 가능성은 사실상 거의 없습니다. 우연은 이 사례에서는 설득력 있는 설명이 아닙니다.


질문	답	지표
"관계가 실질적으로 강한가?"	크다 / 작다	r 값 (효과크기)
"이 관계가 우연이 아니라고 확신할 수 있는가?"	유의하다 / 유의하지 않다	p-값

이 둘은 서로 독립적으로 움직입니다. 즉 네 가지 조합이 모두 가능해요:

크고 + 유의함: r=0.87, p<0.001 → 관계도 강하고, 확신도 있음 (오늘 사례)
크지만 + 유의하지 않음: r=0.6인데 n=5 (표본이 너무 작아서) → 강해 보이는데 우연일 가능성을 배제 못함
작지만 + 유의함: r=0.02, n=100,000 → 관계는 미미한데 표본이 워낙 커서 "0은 아니다"라고 확신
작고 + 유의하지 않음: r=0.03, n=50 → 관계도 약하고 확신도 없음

## 문제 3 · 그 관계를 하나의 식으로 만들 수 있을까?

`난이도 중` · `예상 25분`

**📖 상황**

- 상관계수는 "관계가 얼마나 **강한가**"만 알려주고, "날개가 1mm 길면 몸무게가 **몇 g** 늘어나는가"는
  말해주지 않습니다. 그 답을 주는 것이 **선형회귀(Linear Regression)**입니다.

- 흩어진 점들을 가장 잘 가로지르는 직선 `y = a + bx`를 찾아 관계를 하나의 **식**으로 표현합니다.
  `a`(절편)는 x가 0일 때의 예측값, `b`(기울기)는 **x가 1 늘 때 y가 평균적으로 변하는 양**입니다.

- 그런데 상관과 회귀는 **서로 남이 아닙니다.** 문제 2에서 구한 `r`과
  이번에 구할 기울기·결정계수 사이에는 정확한 관계식이 성립합니다.

**🎯 이 문제로 배우는 것**

- 회귀식을 적합·해석하고, **상관과 회귀가 한 뿌리임을 숫자로 확인**합니다.

In [ ]:
# 문제 3 · 데이터 준비 — 실행만 하세요
penguins_cleaned = penguins.dropna(subset=['body_mass_g', 'flipper_length_mm'])

print("분석에 사용할 데이터:", len(penguins_cleaned), "행")
print("문제 2의 df_corr과 동일한가?:", len(penguins_cleaned) == len(df_corr))

### Q1 · 단순회귀 모델을 적합해 봅시다

- **날개 길이(설명변수) → 몸무게(반응변수)** 회귀 모델을 적합하세요.

- 적합한 모델은 반드시 `model`이라는 이름으로 저장하세요. **문제 4·5에서 이어서 사용합니다.**

- `💡 힌트` `model = smf.ols(formula='body_mass_g ~ flipper_length_mm', data=...).fit()`

In [ ]:
# 문제 3 · Q1
# 여기에 코드를 작성해주세요.
import statsmodels.formula.api as smf

model = smf.ols(formula='body_mass_g ~ flipper_length_mm', data=penguins_cleaned).fit()

print(model.summary())


### Q2 · 분석 결과표를 출력해 봅시다

- 적합한 모델의 요약 결과표를 출력하세요.

- `coef`, `P>|t|`, `R-squared` 열이 각각 어디에 있는지 눈으로 찾아 두세요. 다음 문제들에서 계속 씁니다.

- `💡 힌트` 모델 객체의 `.summary()` 메소드를 사용하세요.

In [ ]:
# 문제 3 · Q2
# 여기에 코드를 작성해주세요.
print(model.summary())

# body_mass_g = -5780.83 + 49.69 × flipper_length_mm

### Q3 · 회귀식을 완성해 봅시다

- Q2의 결과표에서 **절편**과 **기울기**를 찾아 오른쪽 빈칸을 채우세요.

- 숫자를 옮겨 적은 뒤, 완성된 식을 한 줄로 써 보세요.

In [ ]:
# 문제 3 · Q3
# 절편(a) 값:-5780.83
# 기울기(b) 값:49.69
# 완성된 회귀식 → 몸무게(g) = -5780.83 +49.69 × 날개길이(mm)

### Q4 · 회귀식으로 예측해 봅시다

- 날개 길이가 평균보다 **10mm 더 긴** 펭귄은, 평균적인 펭귄보다 몸무게가 몇 g 더 무거울까요?

- 계산 결과를 단위(g)와 함께 출력하세요.

- `💡 힌트` 기울기 × 10

In [ ]:
# 문제 3 · Q4
# 여기에 코드를 작성해주세요.
slope = model.params['flipper_length_mm']
weight_diff = slope * 10

print(f"날개 길이가 평균보다 10mm 더 긴 펭귄은,")
print(f"평균적인 펭귄보다 몸무게가 약 {weight_diff:.2f}g 더 무거울 것으로 예측된다.")

### Q5 · 상관과 회귀를 잇는 두 관계식을 확인해 봅시다

- 다음 두 식이 실제로 성립하는지 **숫자로** 확인하세요.
  - ① 기울기 = `r` × (y의 표준편차 / x의 표준편차)
  - ② 결정계수 `R²` = `r²` _(단순회귀에서만 성립)_

- 좌변과 우변을 **나란히 출력**해 두 값이 일치하는지 눈으로 확인하세요.

- `💡 힌트` 문제 2 Q2의 `r`을 사용합니다. 표준편차는 `.std(ddof=1)`, 모델의 R²는 `model.rsquared`입니다.

In [ ]:
# 문제 3 · Q5
# 여기에 코드를 작성해주세요.
# ① 기울기 = r × (y의 표준편차 / x의 표준편차)
std_x = penguins_cleaned['flipper_length_mm'].std(ddof=1)
std_y = penguins_cleaned['body_mass_g'].std(ddof=1)

slope_from_r = r * (std_y / std_x)
slope_from_model = model.params['flipper_length_mm']

print("① 기울기 관계식 확인")
print(f"  좌변 (model의 기울기)        = {slope_from_model:.4f}")
print(f"  우변 (r × std_y/std_x)       = {slope_from_r:.4f}")
print(f"  일치 여부: {np.isclose(slope_from_model, slope_from_r)}")

print()

# ② R² = r²
r_squared_from_r = r ** 2
r_squared_from_model = model.rsquared

print("② 결정계수 관계식 확인")
print(f"  좌변 (model의 R²)   = {r_squared_from_model:.4f}")
print(f"  우변 (r²)           = {r_squared_from_r:.4f}")
print(f"  일치 여부: {np.isclose(r_squared_from_model, r_squared_from_r)}")

#① 번에서 비교한 것

#두 개의 서로 다른 계산 방법으로 구한 기울기 값을 비교했습니다.

#좌변: model.params['flipper_length_mm'] → statsmodels가 **회귀분석(최소제곱법)**으로 직접 계산해준 기울기값
#우변: r × (std_y / std_x) → 우리가 상관계수 공식을 이용해서 손으로(공식으로) 계산한 값

#이 두 값은 완전히 다른 계산 과정을 거쳐 나온 숫자입니다. 하나는 회귀모델이 데이터를 직접 적합해서 뽑아낸 값이고, 다른 하나는 상관계수 r과 표준편차라는 완전히 다른 재료로 계산한 값이에요. 근데 계산해보니 49.6856 = 49.6856으로 정확히 같았다는 게 포인트입니다.

#왜 이걸 확인했을까요?

#"어, 회귀분석이랑 상관분석이 완전히 다른 도구인 줄 알았는데, 사실 같은 수학적 뿌리에서 나온 거였네?"를 눈으로 확인시켜주기 위해서예요. 즉:

#"기울기라는 걸 굳이 회귀분석을 따로 돌리지 않고, 이미 알고 있는 r(상관계수)과 표준편차만 가지고도 정확히 계산해낼 수 있다"

#는 걸 증명한 겁니다. 이게 두 분석이 "완전히 별개의 도구"가 아니라 "하나의 관계를 표현하는 두 가지 방식"이라는 걸 보여주는 증거예요.

#② 번도 마찬가지 구조입니다

#좌변: model.rsquared → 회귀모델이 직접 계산해준 결정계수
#우변: r ** 2 → 상관계수 r을 그냥 제곱만 한 값

#이것도 서로 완전히 다른 경로로 계산했는데 값이 정확히 같았죠 (0.759 = 0.759).

#정리하면

#두 문항 모두 "회귀분석 결과" vs **"상관계수로부터 유도한 값"**을 비교한 거예요. 다른 방법으로 계산했는데 똑같은 숫자가 나온다는 걸 직접 확인함으로써, "상관과 회귀는 수학적으로 연결되어 있다(같은 뿌리다)"는 걸 증명하는 실습이었습니다.

### 💬 정리 · 결과를 말로 설명해 보기

- 기울기(b) 값을 "날개 길이가 ○○할 때 몸무게가 ○○한다"는 문장으로 풀어 써 보세요.

- 절편(a)이 음수로 나왔습니다. 이 숫자를 "날개 길이가 0mm인 펭귄의 몸무게"로 해석해도 될까요? 왜 그럴까요?

- Q5에서 `R² = r²`을 확인했습니다. 그렇다면 설명변수와 반응변수를 서로 바꿔 분석하면 **기울기**와 **R²**는 각각 같을까요, 다를까요?

- 상관계수 `r`은 단위 없는 −1~1 값이고, 기울기 `b`는 'mm당 g'이라는 단위를 가집니다. 회귀가 상관보다 더 알려주는 것은 무엇인가요?

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- 절편은 **데이터가 존재하는 범위 밖의 값**입니다. 날개 길이 0mm인 펭귄은 우리 데이터에 없습니다 — 회귀식은 관측 범위 안에서만 의미를 갖습니다.

- x와 y를 바꿨을 때를 생각할 때 기준은 **"r은 대칭인가, s_y/s_x는 대칭인가"**입니다. 두 관계식 ①②를 각각 대입해 보면 답이 갈립니다.

- "강도"와 "변화량"의 차이로 정리해 보세요. 보고서에 "1mm당 ○g"을 쓸 수 있는 쪽은 어느 것인가요?

</details>

In [ ]:
# 문제 3 · 정리
# 여기에 의견을 작성해주세요.
"날개 길이가 1mm 길어질 때, 몸무게는 평균적으로 49.69g 무거워진다."

핵심은 "평균적으로"라는 표현이에요. 회귀직선은 모든 개체에 정확히 들어맞는 게 아니라, 전체적인 경향을 나타내는 것이므로 개별 펭귄마다 정확히 49.69g씩 차이 나는 건 아닙니다.

안 됩니다. 이유는 두 가지입니다.

외삽(extrapolation)의 문제: 우리 데이터는 날개 길이가 대략 170~230mm 범위에서만 관측되었습니다. 회귀직선은 이 관측된 범위 안에서만 신뢰할 수 있고, 범위를 훨씬 벗어난 0mm 지점까지 직선을 그대로 연장해서 해석하는 건 근거가 없습니다.
생물학적으로 말이 안 됨: 날개 길이가 0mm인 펭귄은 애초에 존재할 수 없고, 몸무게가 음수(-5780g)라는 것도 현실적으로 불가능합니다.

    절편은 그저 회귀직선을 수학적으로 정의하기 위해 필요한 값(y축과 만나는 점)일 뿐, 이 사례에서는 그 자체로 실질적인 의미를 갖지 않습니다.

R²는 같습니다. R² = r²인데, 상관계수 r은 x와 y의 역할을 바꿔도 값이 동일합니다 (corr(x,y) = corr(y,x)). 그래서 어느 쪽을 설명변수로 두든 R²는 0.759로 동일합니다.
기울기는 다릅니다. 기울기 = r × (반응변수의 표준편차 / 설명변수의 표준편차)인데, 분모와 분자가 바뀌면 값도 달라집니다. flipper_length_mm ~ body_mass_g로 반대로 돌리면 기울기는 전혀 다른 숫자가 나올 겁니다 (역수 관계도 아닙니다. 정확히는 b_xy × b_yx = r²라는 관계가 성립합니다).

r은 "관계가 얼마나 강한가/약한가"라는 강도만 알려줍니다. 반면 회귀는:

구체적인 예측을 가능하게 합니다 — "날개 길이가 몇 mm면 몸무게는 몇 g일까?"라는 실제 수치를 계산할 수 있습니다.
실용적 단위를 가진 변화율을 알려줍니다 — "1mm 늘 때 몇 g이 느는가"라는, 현실에서 바로 쓸 수 있는 정보입니다.
r은 두 변수 사이의 대칭적인 관계(누가 원인이고 결과인지 구분 안 함)만 말하지만, 회귀는 "이 변수를 알면 저 변수를 예측할 수 있다"는 방향성 있는 모델을 제공합니다.

한마디로, 상관은 "관계가 있다/강하다"는 진단이고, 회귀는 "그 관계를 이용해 실제로 예측하고 설명하는" 도구라고 볼 수 있습니다.

## 문제 4 · 이 기울기, 우연이 아니라고 말할 수 있을까?

`난이도 중` · `예상 25분`

**📖 상황**

- 문제 3에서 기울기를 구했습니다. 그런데 이 값은 **표본 342마리에서 계산된 값**입니다.

- 만약 전체 펭귄 집단에서 두 변수가 실제로는 아무 관계가 없다면(**기울기 = 0**),
  우리가 얻은 기울기는 그저 **표본을 뽑는 과정에서 우연히 생긴 값**일 수도 있습니다.

- 이 의심에 답하는 것이 **회귀계수의 가설검정**이고, 결과표의 `P>|t|` 열이 판단 근거입니다.

**🎯 이 문제로 배우는 것**

- 강의에서 배운 **상관계수의 유의성 검정**과 회귀계수 검정이 **같은 검정**임을 확인하고,
  p-값이 항상 작게 나오는 것은 아니라는 사실을 **기각 실패 사례**로 직접 만납니다.

In [ ]:
# 문제 4 · 데이터 준비 — 실행만 하세요
# 문제 3의 Q1을 먼저 완료해야 이 문제를 풀 수 있습니다.

print("분석 대상 데이터:", len(penguins_cleaned), "행 (문제 3과 동일)")
print("설명변수: flipper_length_mm  /  반응변수: body_mass_g")

### Q1 · 가설을 세워 봅시다

- 회귀계수(기울기)에 대한 귀무가설(H₀)과 대립가설(H₁)을 오른쪽 빈칸에 쓰세요.

- '기울기'가 무엇과 같은지/다른지를 명확히 적으세요.

In [ ]:
# 문제 4 · Q1
# H₀ (귀무가설): 기울기가 0 이다(모집단에서 날개 길이는 몸무게와 관계가 없다)
# H₁ (대립가설): 기울기가 0이 아니다(모집단에서 날개 길이는 몸무게와 관계가 있다)

### Q2 · 기울기의 p-값을 찾아 봅시다

- `flipper_length_mm` 행에서 `P>|t|` 열의 값을 찾아 출력하세요.

- `💡 힌트` 결과표에서 눈으로 찾아도 되고, 모델 객체의 `.pvalues` 속성을 써도 됩니다.

In [ ]:
# 문제 4 · Q2
# 여기에 코드를 작성해주세요.
p_value_slope = model.pvalues['flipper_length_mm']
print(f"flipper_length_mm의 p-값 (P>|t|): {p_value_slope}")

### Q3 · 유의수준 0.05로 결론을 내려 봅시다

- p-값과 유의수준 `alpha = 0.05`를 비교해, **판정 결과를 문장으로 출력하는 코드**를 작성하세요.

- 숫자만 찍지 말고 "기각한다 / 기각하지 못한다"까지 출력되도록 만드세요.

In [ ]:
# 문제 4 · Q3
# alpha = 0.05

# 여기에 코드를 작성해주세요.
alpha = 0.05

print(f"p-값 = {p_value_slope}")
print(f"유의수준 alpha = {alpha}")

if p_value_slope < alpha:
    print(f"\np-값이 alpha({alpha})보다 작으므로 귀무가설(H0: 기울기=0)을 기각한다.")
    print("→ 날개 길이는 몸무게와 통계적으로 유의한 관계가 있다.")
else:
    print(f"\np-값이 alpha({alpha})보다 크므로 귀무가설(H0: 기울기=0)을 기각하지 못한다.")
    print("→ 날개 길이와 몸무게 사이에 유의한 관계가 있다고 보기 어렵다.")

### Q4 · 상관계수의 검정과 같은 검정인지 확인해 봅시다

- 강의에서 배운 공식으로 t-통계량을 **직접 계산**하세요.

  `t = r × √(n − 2) / √(1 − r²)`

- 이 값을 `model.tvalues['flipper_length_mm']`와 **나란히 출력**해 비교하세요.

- `💡 힌트` `n`은 표본 크기(`len(penguins_cleaned)`), `r`은 문제 2 Q2에서 구한 값입니다.

In [ ]:
# 문제 4 · Q4
# 여기에 코드를 작성해주세요.
n = len(penguins_cleaned)

t_from_formula = r * np.sqrt(n - 2) / np.sqrt(1 - r**2)
t_from_model = model.tvalues['flipper_length_mm']

print(f"공식으로 직접 계산한 t값     = {t_from_formula:.4f}")
print(f"model.tvalues의 t값         = {t_from_model:.4f}")
print(f"일치 여부: {np.isclose(t_from_formula, t_from_model)}")

#하나는 상관계수 r을 이용한 공식으로 계산했고, 다른 하나는 회귀모델(statsmodels)이 기울기의 유의성을 검정하기 위해 계산한 t값입니다. 완전히 다른 두 경로로 계산했는데 정확히 같은 숫자가 나왔다는 건:

#"상관계수가 0인지 검정하는 것"과 "회귀계수(기울기)가 0인지 검정하는 것은 사실 같은 검정이다"

### Q5 · p-값이 크게 나오는 사례를 만나 봅시다

- **Adelie 종 수컷만** 골라, `bill_length_mm`으로 `body_mass_g`를 설명하는 회귀를 적합하세요.

- 기울기의 p-값을 유의수준 0.05와 비교하고, 표본 크기(n)와 `R²`도 함께 출력하세요.

- `💡 힌트` 먼저 필요한 열의 결측치를 제거한 뒤 조건으로 걸러 냅니다.
  `df[(df['species'] == 'Adelie') & (df['sex'] == 'Male')]`

In [ ]:
# 문제 4 · Q5
# 여기에 코드를 작성해주세요.
# 필요한 열의 결측치 제거 후 조건으로 필터링
df_adelie_male = penguins.dropna(subset=['bill_length_mm', 'body_mass_g', 'species', 'sex'])
df_adelie_male = df_adelie_male[(df_adelie_male['species'] == 'Adelie') & (df_adelie_male['sex'] == 'Male')]

print("Adelie 수컷 표본 크기:", len(df_adelie_male))

# 회귀 적합
model_adelie = smf.ols(formula='body_mass_g ~ bill_length_mm', data=df_adelie_male).fit()

n_adelie = len(df_adelie_male)
p_value_adelie = model_adelie.pvalues['bill_length_mm']
r_squared_adelie = model_adelie.rsquared
slope_adelie = model_adelie.params['bill_length_mm']

alpha = 0.05

print(f"\n표본 크기 n = {n_adelie}")
print(f"기울기 = {slope_adelie:.4f}")
print(f"R-squared = {r_squared_adelie:.4f}")
print(f"p-값 = {p_value_adelie:.4f}")

if p_value_adelie < alpha:
    print(f"\np-값이 {alpha}보다 작으므로 귀무가설(기울기=0)을 기각한다.")
    print("→ Adelie 수컷에서 부리 길이는 몸무게와 유의한 관계가 있다.")
else:
    print(f"\np-값이 {alpha}보다 크므로 귀무가설(기울기=0)을 기각하지 못한다.")
    print("→ Adelie 수컷에서 부리 길이와 몸무게 사이에 유의한 관계가 있다고 보기 어렵다.")

### 💬 정리 · 결과를 말로 설명해 보기

- p-값이 작다는 것은 정확히 무엇이 작다는 뜻인가요? "귀무가설이 참일 확률"이라고 말해도 될까요?

- Q4에서 두 t값이 일치했습니다. 이것은 상관계수의 검정과 회귀계수의 검정이 **어떤 관계**임을 뜻하나요?

- Q5의 결과는 Q2와 어떻게 달랐나요? 같은 `penguins` 데이터인데 결론이 갈린 이유를 **표본 크기**와 **관계의 강도(R²)** 두 측면에서 설명해 보세요.

- Q5처럼 p-값이 0.05보다 조금 큰 경우, "두 변수는 관계가 없음이 증명되었다"고 말할 수 있을까요? 어떻게 말하는 것이 정확할까요?

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- p-값은 "H₀가 참이라고 **가정했을 때** 이런 결과가 나올 확률"입니다. 가정과 결론의 방향을 뒤집지 않도록 문장을 조심해서 써 보세요.

- Q4의 일치는 우연이 아닙니다. 단순회귀에서는 "두 변수가 무관하다"는 하나의 가설을 두 가지 방식으로 적었을 뿐입니다 — 그래서 t도 p도 같습니다.

- 기각 실패는 **"관계가 없다"가 아니라 "관계가 있다는 증거가 부족하다"**입니다. Q5의 n과 R²를 보면 왜 증거가 부족했는지 짐작할 수 있습니다.

</details>

In [ ]:
# 문제 4 · 정리
# 여기에 의견을 작성해주세요.

1. 안 됩니다. 이건 통계에서 가장 흔한 오해 중 하나예요.

p-값의 정확한 정의는: **"귀무가설(H₀)이 참이라고 가정했을 때, 우리가 관측한 것만큼 극단적인(또는 더 극단적인) 결과가 나올 확률"**입니다.

즉 p-값은 데이터에 대한 확률이지, 가설에 대한 확률이 아닙니다.

❌ "귀무가설이 참일 확률이 4.37e-107이다" (틀림 — 이건 가설 자체의 확률처럼 말하는 것)
✅ "귀무가설이 참이라면(즉 진짜 관계가 없다면), 우리가 얻은 것 같은 강한 관계가 우연히 나올 확률이 4.37e-107이다" (맞음)

p(귀무가설이 참) 이라는 확률은 베이즈 통계의 영역이고, 우리가 지금 쓰는 빈도주의적 p-값과는 다른 개념입니다. 이 둘을 혼동하는 게 실무에서도 아주 흔한 오류이니 확실히 구분해두시는 게 좋습니다.

2. 상관계수(r)가 0인지 검정하는 것과, 단순회귀에서 기울기(β)가 0인지 검정하는 것이 수학적으로 완전히 동일한 검정이라는 뜻입니다. 계산 경로(r 기반 공식 vs 회귀모델의 t검정)는 다르지만, 결국 같은 질문("두 변수 사이에 선형 관계가 있는가?")에 대한 답이기 때문에 t값도 정확히 같게 나옵니다. 단순회귀에서는 상관분석과 회귀분석이 서로 다른 도구가 아니라 하나의 관계를 표현하는 두 가지 언어라는 걸 다시 한번 확인시켜주는 결과입니다.

3.    	Q2 (전체)	Q5 (Adelie 수컷)
n	342	73
R²	0.759	0.049
p-값	≈0	0.061
표본 크기: n이 342 → 73으로 5분의 1 수준으로 줄었습니다. t값 공식(t = r√(n-2)/√(1-r²))에서 분자의 √(n-2)가 작아지면 t값도 작아지고, p-값은 커집니다.
관계의 강도: R²가 0.759 → 0.049로 급격히 떨어졌습니다. 전체 데이터에서 보였던 강한 관계의 상당 부분은 사실 종과 성별 간 차이(Gentoo는 크고 무겁고 Adelie는 작다는 식의 집단 차이) 때문이었을 가능성이 큽니다. 같은 종·같은 성별로 좁혀서 "순수한 개체 간 관계"만 보니 관계가 훨씬 약해진 거죠.

작은 표본과 약한 관계, 이 두 가지 불리한 조건이 겹치면서 p-값이 0.05를 살짝 넘긴 겁니다.

4. 말할 수 없습니다. 이건 통계적 가설검정의 근본적인 비대칭성 때문입니다.

가설검정은 애초에 **"관계가 있다는 것을 적극적으로 증명"**하는 데는 쓸 수 있어도, **"관계가 없다는 것을 증명"**하는 데는 쓸 수 없는 구조입니다.
"기각하지 못했다"는 건 "관계가 없다는 게 확인됐다"가 아니라, **"이 표본으로는 관계가 있다고 확신할 만큼 충분한 증거를 찾지 못했다"**는 뜻일 뿐입니다.
특히 지금처럼 표본이 작을 때(n=73)는, 실제로 관계가 존재하더라도 그걸 통계적으로 검출할 만한 힘(statistical power)이 부족했을 가능성이 큽니다.


## 문제 5 · 미니 프로젝트 — 이 회귀모형, 믿고 보고해도 될까?

`난이도 상` · `예상 35분`

**📖 상황**

- 계수가 유의하다고 해서 분석이 끝난 것은 아닙니다. 보고서를 쓰기 전에 두 가지를 더 확인합니다.

- **① 얼마나 설명하는가 — 결정계수(R²)**
  반응변수 전체 변동 중 이 모델이 설명한 비율입니다.

- **② 전제가 지켜졌는가 — 잔차(Residual) 진단**
  잔차는 `실제값 − 예측값`입니다. 좋은 모델이라면 잔차가 **패턴 없이 0 주위에 고르게** 흩어져야 합니다.
  깔때기(부채꼴) 모양이면 **등분산성**이 깨졌다는 신호이고, 이때는 p-값과 신뢰구간을 그대로 믿기 어렵습니다.

**🎯 이 문제로 배우는 것**

- 우리 모형을 진단한 뒤, **일부러 문제가 있는 다른 모형과 나란히 놓고 비교**합니다.
  "통과한 잔차"만 보면 진단을 왜 하는지 알 수 없기 때문입니다.

In [ ]:
# 문제 5 · 데이터 준비 — 실행만 하세요
# 문제 3의 Q1을 먼저 완료해야 이 문제를 풀 수 있습니다.

print("분석 대상 데이터:", len(penguins_cleaned), "행")
print("모형: body_mass_g ~ flipper_length_mm")

### Q1 · 결정계수로 설명력을 평가해 봅시다

- 모델의 `R-squared` 값을 출력하세요.

- 그 값이 뜻하는 바를 "몸무게 변동의 몇 %" 형태로 오른쪽 빈칸에 쓰세요.

- `💡 힌트` `model.rsquared`

In [ ]:
# 문제 5 · Q1
# R-squared 값: 0.7590
# 설명력 해석 (몸무게 변동의 몇 %를 설명하나요?): 몸무게(body_mass_g) 변동의 약 75.9%를 날개 길이(flipper_length_mm) 하나로 설명한다

# 여기에 코드를 작성해주세요.
r_squared = model.rsquared
print(f"R-squared = {r_squared:.4f}")
print(f"→ 몸무게(body_mass_g) 변동의 약 {r_squared*100:.1f}%를 날개 길이(flipper_length_mm) 하나로 설명한다.")


### Q2 · 예측값과 잔차를 계산해 봅시다

- 모델의 **예측값**과 **잔차**를 각각 계산해 변수에 저장하세요.

- 잔차의 개수와 평균을 함께 출력해, 잔차가 0 주위에 모여 있는지 확인하세요.

- `💡 힌트` 예측값은 `.fittedvalues`, 잔차는 `.resid` 속성입니다.

In [ ]:
# 문제 5 · Q2
# 여기에 코드를 작성해주세요.

fitted_values = model.fittedvalues
residuals = model.resid

print(f"잔차의 개수: {len(residuals)}")
print(f"잔차의 평균: {residuals.mean():.10f}")

### Q3 · 잔차 산점도를 그려 봅시다

- 가로축을 **예측값**, 세로축을 **잔차**로 하는 산점도를 그리세요.

- `y = 0` 기준선을 함께 표시하면 패턴을 보기 쉽습니다.

- `💡 힌트` `plt.scatter()` 와 `plt.axhline(0, color='red', linestyle='--')`

In [ ]:
# 문제 5 · Q3
# 여기에 코드를 작성해주세요.
plt.figure(figsize=(8, 6))
plt.scatter(fitted_values, residuals, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.title('잔차 산점도 (예측값 vs 잔차)')
plt.xlabel('예측값 (fitted values)')
plt.ylabel('잔차 (residuals)')
plt.tight_layout()

### Q4 · 등분산성을 숫자로 검정해 봅시다

- **브루쉬-페이건 검정**으로 등분산성을 확인하세요.

- H₀는 "등분산이다"입니다. p-값이 0.05보다 작으면 이분산을 의심합니다.

- 판정 결과를 문장으로 함께 출력하세요.

- `💡 힌트` `het_breuschpagan(잔차, model.model.exog)`는
  `(LM통계량, LM p-값, F통계량, F p-값)`을 돌려줍니다.

In [ ]:
# 문제 5 · Q4
# 여기에 코드를 작성해주세요.
from statsmodels.stats.diagnostic import het_breuschpagan

lm_stat, lm_pvalue, f_stat, f_pvalue = het_breuschpagan(residuals, model.model.exog)

print(f"LM 통계량 = {lm_stat:.4f}")
print(f"LM p-값   = {lm_pvalue:.4f}")

alpha = 0.05
if lm_pvalue < alpha:
    print(f"\np-값이 {alpha}보다 작으므로 귀무가설(등분산이다)을 기각한다.")
    print("→ 이분산성이 의심된다. 등분산성 가정이 깨졌을 수 있다.")
else:
    print(f"\np-값이 {alpha}보다 크므로 귀무가설(등분산이다)을 기각하지 못한다.")
    print("→ 등분산성 가정에 문제가 없다고 볼 수 있다.")

### Q5 · 문제가 있는 모형과 나란히 비교해 봅시다

- **비교군**으로 `body_mass_g ~ bill_length_mm` 모형을 적합해 `model_bill`로 저장하세요.

- 두 모형의 잔차 산점도를 **나란히** 그리세요.

- 두 모형의 브루쉬-페이건 p-값을 **함께 출력**해 비교하세요.

- `💡 힌트` `fig, ax = plt.subplots(1, 2, figsize=(12, 4))`로 두 그림을 옆으로 배치할 수 있습니다.

In [ ]:
# 문제 5 · Q5
# 여기에 코드를 작성해주세요.
# 비교군 모형 적합
model_bill = smf.ols(formula='body_mass_g ~ bill_length_mm', data=penguins_cleaned).fit()

fitted_bill = model_bill.fittedvalues
residuals_bill = model_bill.resid

# 두 모형의 잔차 산점도를 나란히 그리기
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].scatter(fitted_values, residuals, alpha=0.6)
ax[0].axhline(0, color='red', linestyle='--')
ax[0].set_title('model: body_mass_g ~ flipper_length_mm')
ax[0].set_xlabel('예측값')
ax[0].set_ylabel('잔차')

ax[1].scatter(fitted_bill, residuals_bill, alpha=0.6)
ax[1].axhline(0, color='red', linestyle='--')
ax[1].set_title('model_bill: body_mass_g ~ bill_length_mm')
ax[1].set_xlabel('예측값')
ax[1].set_ylabel('잔차')

plt.tight_layout()

# 브루쉬-페이건 p-값 비교
_, p_flipper, _, _ = het_breuschpagan(residuals, model.model.exog)
_, p_bill, _, _ = het_breuschpagan(residuals_bill, model_bill.model.exog)

print(f"model (flipper_length_mm) 의 Breusch-Pagan p-값 = {p_flipper:.4f}")
print(f"model_bill (bill_length_mm) 의 Breusch-Pagan p-값 = {p_bill:.4f}")

### Q6 · 보고 문장으로 정리해 봅시다

- 지금까지의 결과를 종합해, 우리 회귀모형을 보고서에 어떻게 적을지 **한 문장**으로 쓰세요.

- **기울기 + 유의성 + 설명력 + 진단 결과**를 모두 담아 보세요.

In [ ]:
# 문제 5 · Q6
# 보고 문장:
"날개 길이(flipper_length_mm)는 몸무게(body_mass_g)와 통계적으로 유의한 양의 관계를 가지며(β = 49.69g/mm, p < 0.001), 날개 길이 하나만으로 몸무게 변동의 약 75.9%(R² = 0.759)를 설명할 수 있고, 잔차 진단 결과 등분산성 가정도 충족되어(Breusch-Pagan p = 0.142) 이 회귀 결과를 신뢰할 수 있다."

각 요소가 왜 들어갔는지 짚어보면

기울기(β = 49.69): "날개 길이가 1mm 늘 때 몸무게가 얼마나 느는가"라는 실용적 크기
유의성(p < 0.001): 이 관계가 우연이 아니라는 통계적 근거
설명력(R² = 0.759): 이 모델이 몸무게 변동을 얼마나 잘 설명하는가
진단(Breusch-Pagan p = 0.142): 이 p-값과 신뢰구간을 그대로 믿어도 되는가에 대한 확인

### 💬 정리 · 결과를 말로 설명해 보기

- Q3·Q4에서 우리 모형의 잔차는 어떤 상태였나요? 이 결과를 근거로 문제 3·4의 해석을 신뢰할 수 있을까요?

- Q5의 두 잔차 산점도는 어떻게 달랐나요? 비교군에서 관찰된 모양을 강의에서 배운 용어로 무엇이라 부르나요?

- 비교군처럼 등분산성이 깨졌을 때, **잘못된 것은 기울기 값 자체인가요, 아니면 그 값의 통계적 해석인가요?**

- R²가 1이 아니라는 것은 설명되지 않은 변동이 남아 있다는 뜻입니다. `penguins` 안에서 몸무게에 영향을 줄 만한데 우리 모델이 놓치고 있는 변수는 무엇일까요?

- 그 변수들을 모델에 함께 넣으려면 어떤 분석이 필요할까요?

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- 강의 7장은 이렇게 적었습니다 — _"파라미터 값이 잘못되었다는 뜻이 아니라, 이 값의 통계적 해석(유의성·신뢰구간)과 예측 결과를 신뢰할 수 없다는 의미"_. 세 번째 질문의 답이 여기 있습니다.

- 비교군의 잔차가 퍼지는 모양을 강의의 3케이스 그림(정규성 O·등분산 O / X·O / O·X)과 맞춰 보세요.

- 놓친 변수 후보는 문제 1의 그림에 이미 있습니다. 그 변수를 모델에 함께 넣는 분석의 이름은 다음 8장의 제목입니다.

</details>

In [ ]:
# 문제 5 · 정리
# 여기에 의견을 작성해주세요.

우리 모형(body_mass_g ~ flipper_length_mm)의 잔차는 예측값과 무관하게 고르게, 무작위로 흩어져 있었습니다. 시각적으로도 일정한 폭의 띠 모양이었고, Breusch-Pagan 검정으로도 p=0.1418로 등분산성 가정을 기각하지 못했습니다.

신뢰할 수 있습니다. 등분산성이 지켜졌다는 건, 문제 3에서 구한 기울기(49.69)와 R²(0.759), 문제 4에서 구한 p-값과 유의성 검정 결과가 왜곡 없이 계산되었다는 뜻입니다. 등분산은 회귀분석의 표준오차 계산이 정확하기 위한 전제 조건이기 때문에, 이 조건이 충족되었다는 건 우리가 지금까지 내린 통계적 결론(유의하다, 강한 관계다)을 그대로 믿어도 된다는 근거가 됩니다.

model(flipper)은 예측값에 상관없이 잔차 폭이 일정한 띠 모양이었지만, model_bill(bill)은 예측값이 커질수록 잔차 폭이 눈에 띄게 넓어지는 깔때기(부채꼴) 모양이었습니다. 강의에서 배운 용어로는 이 깔때기 모양이 바로 **이분산성(heteroscedasticity)**의 전형적인 시각적 신호입니다.

통계적 해석(표준오차, p-값, 신뢰구간)이 잘못됩니다. 기울기 값 자체(점추정치)는 여전히 유효합니다.

OLS(최소제곱법)로 구한 기울기는 등분산성 가정이 깨져도 여전히 불편추정량(unbiased estimator)입니다. 즉 "가장 적합한 직선"이라는 점에서는 여전히 최선의 추정값입니다. 문제는 그 기울기의 표준오차 계산이 등분산을 전제로 하고 있다는 것입니다. 이분산 상태에서는 표준오차가 실제보다 과소 또는 과대평가될 수 있고, 그러면 여기서 유도되는 t값, p-값, 신뢰구간이 모두 부정확해집니다. 즉 "얼마나 확신할 수 있는가"에 대한 판단이 흔들리는 것이지, "관계의 방향과 대략적인 크기"자체가 틀렸다는 뜻은 아닙니다.

날개 길이 하나로 몸무게 변동의 75.9%를 설명하지만, 나머지 24.1%는 다른 요인 때문입니다. penguins 데이터 안에서 몸무게에 영향을 줄 만한데 지금 모델에 안 들어간 변수들:

species(종): 문제 1에서 봤듯 Gentoo는 몸집이 크고 무겁고, Adelie는 상대적으로 작습니다. 종 자체가 몸무게에 큰 영향을 줍니다.
sex(성별): 문제 4 Q5에서 Adelie 수컷만 따로 봤을 때 결과가 달랐던 것처럼, 일반적으로 수컷이 암컷보다 체구가 큰 경향이 있습니다.
bill_length_mm, bill_depth_mm: 부리 관련 변수도 어느 정도 몸무게와 관련이 있을 수 있습니다(다만 model_bill에서 봤듯 단독으로는 문제가 있었죠).
island(서식 섬): 서식지에 따른 먹이 환경 차이가 간접적으로 영향을 줄 수도 있습니다.

**다중회귀분석(multiple regression)**이 필요합니다. 예를 들면:
